# Pan-Cancer RNA-seq: Clustering and Cancer Type Prediction
**Author:** Davie Slocum

## Research Questions
1. Do unsupervised clusters derived from tumor gene expression profiles correspond to known cancer types?
2. Can we accurately predict a tumor’s cancer type from its gene expression profile, and which genes are most informative?


## Part A - Load data + sanity checks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Load processed expression data + labels
X = pd.read_csv("../data/expression_processed.csv.gz", index_col=0)
y = pd.read_csv("../data/labels.csv", index_col=0).iloc[:, 0]

print("X shape (samples x genes):", X.shape)
print("Number of classes:", y.nunique())
print(y.value_counts().head(10))

## Part B - Clustering (unsupervised)

## Unsupervised Learning: PCA + K-means Clustering

We reduce dimensionality using PCA and then cluster samples using K-means.
We evaluate cluster quality internally (silhouette score) and externally by comparing clusters to known cancer-type labels (ARI, NMI).

In [ ]:
# PCA for dimensionality reduction
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X)

print("Explained variance by first 10 PCs:", pca.explained_variance_ratio_[:10].round(4))

In [ ]:
# choose K + cluster
k = 20
kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
clusters = kmeans.fit_predict(X_pca)

sil = silhouette_score(X_pca, clusters)
ari = adjusted_rand_score(y, clusters)
nmi = normalized_mutual_info_score(y, clusters)

print("K =", k)
print("Silhouette:", sil)
print("ARI vs cancer type:", ari)
print("NMI vs cancer type:", nmi)

In [ ]:
# Visualize clusters in 2D PCA space
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X)

plt.figure(figsize=(8,6))
plt.scatter(X_2d[:,0], X_2d[:,1], c=clusters, s=8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means clusters in 2D PCA space")
plt.show()

## Part C - Supervised Learning (predict cancer type)

## Supervised Learning: Predict Cancer Type From Expression

We train a multi-class classifier to predict cancer type from gene expression.
We use a stratified train/test split and report accuracy and macro F1.

In [ ]:
# encode labels + train/test split
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print(X_train.shape, X_test.shape)

In [ ]:
# losgistic regression model
clf = LogisticRegression(
    max_iter=5000,
    multi_class="multinomial",
    n_jobs=-1
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print("Accuracy:", acc)
print("Macro F1:", f1)

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots(figsize=(10,10))
disp.plot(ax=ax, colorbar=False)
plt.title("Confusion Matrix (all classes)")
plt.show()

## Results & Interpretation

### Clustering
- Silhouette score:
- ARI/NMI vs cancer type:
- Interpretation: (Did clusters recover cancer types? Which patterns were visible? Why might some types mix?)

### Supervised Learning
- Accuracy:
- Macro F1:
- Interpretation: (Was prediction strong? Which cancer types were confused? Why?)

### Limitations
- Gene expression is high-dimensional; PCA may lose signal.
- Cancer types can overlap biologically (shared pathways).
- We did not validate on an external dataset.

### Next Steps
- Try alternative clustering (hierarchical, GMM).
- Tune model hyperparameters.
- Explore feature importance to identify key genes.